# mixture similarity
## Build a 100-row, class-balanced prompt CSV from
## 

In [1]:
import sys
import numpy as np
import pandas as pd

IN_CSV  = "Data/training_pairs_discretization.csv"
OUT_CSV = "mixture_similarity_100.csv"

RANDOM_SEED = 1337
TARGET_PER_CLASS = 25
MAX_N = 50  # safety cap

LABEL_ORDER = [
    "Strongly Similar",
    "Slightly Similar",
    "Slightly Dissimilar",
    "Strongly Dissimilar",
]

REQ_COLS = [
    "compound.name_1", "SMILES_1",
    "compound.name_2", "SMILES_2",
    "CID_count_1", "CID_count_2",
    "answer", "Experimental Values"
]

def main():
    rng = np.random.default_rng(RANDOM_SEED)

    # Load
    try:
        df = pd.read_csv(
            IN_CSV,
            dtype=str,
            engine="python",
            keep_default_na=False,
            na_values=[]
        )
    except FileNotFoundError:
        sys.exit(f"Could not find input file: {IN_CSV}")

    # Ensure required columns exist
    missing = [c for c in REQ_COLS if c not in df.columns]
    if missing:
        sys.exit(f"Missing required columns in input: {missing}")

    # Coerce counts & experimental values
    for c in ["CID_count_1", "CID_count_2"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df["Experimental Values"] = pd.to_numeric(df["Experimental Values"], errors="coerce")

    # Basic cleaning
    for c in ["compound.name_1", "SMILES_1", "compound.name_2", "SMILES_2", "answer"]:
        df[c] = (df[c].astype(str)
                      .str.replace("\u00A0", " ", regex=False)
                      .str.strip())

    # Keep valid rows
    base = df.dropna(subset=["CID_count_1", "CID_count_2", "Experimental Values"]).copy()
    base = base[base["answer"].isin(LABEL_ORDER)].copy()
    if base.empty:
        sys.exit("No valid rows after cleaning.")

    # OPTIONS from labels present (respect LABEL_ORDER)
    present = [lbl for lbl in LABEL_ORDER if lbl in base["answer"].unique()]
    if not present:
        sys.exit("No recognized labels found in 'answer' column.")
    options_str = "; ".join(present)

    # Try to gather 25 per class under count constraint; increase n if needed
    n_extra = 0
    selected_parts = None
    while n_extra <= MAX_N:
        upper = 10 + n_extra
        constrained = base[
            (base["CID_count_1"] > 1) & (base["CID_count_1"] <= upper) &
            (base["CID_count_2"] > 1) & (base["CID_count_2"] <= upper)
        ].copy()

        if constrained.empty:
            n_extra += 1
            continue

        buckets = {}
        ok = True
        for lbl in LABEL_ORDER:
            pool = constrained[constrained["answer"] == lbl]
            if len(pool) >= TARGET_PER_CLASS:
                idx = rng.choice(pool.index.to_numpy(), size=TARGET_PER_CLASS, replace=False)
                buckets[lbl] = pool.loc[idx]
            else:
                ok = False
                break

        if ok:
            selected_parts = [buckets[l] for l in LABEL_ORDER]
            print(f"Achieved 25 per class with n = {n_extra} (max CID_count allowed = {upper}).")
            break

        n_extra += 1

    if selected_parts is None:
        sys.exit(f"Could not reach 25 per class even with n up to {MAX_N}.")

    selected = pd.concat(selected_parts, axis=0, ignore_index=True)

                
    # Build prompts (row-wise)
    def make_prompt1(row):
        return (
            f"Mixture A contains molecules [{row['SMILES_1']}] and mixture B contains "
            f"[{row['SMILES_2']}]. On the scale [{options_str}], select how similar these mixtures smell. "
            "If you had to rate the olfactory perceptual distance on a 0.00–1.00 scale (0.00 = identical, 1.00 = completely different)"
            ", what distance do you assign? "
            f"Respond with your selection from [{options_str}], followed by the distance value with two-decimal precision. "
            "Use semicolons (;) as separators. Do not write any comments."
        )

    def make_prompt2(row):
        return (
            f"Mixture A contains molecules [{row['compound.name_1']}] and mixture B contains "
            f"[{row['compound.name_2']}]. On the scale [{options_str}], select how similar these mixtures smell. "
            "If you had to rate the olfactory perceptual distance on a 0.00–1.00 scale (0.00 = identical, 1.00 = completely different)"
            ", what distance do you assign? "
            f"Respond with your selection from [{options_str}], followed by the distance value with two-decimal precision. "
            "Use semicolons (;) as separators. Do not write any comments."
        )


    # Build other_info as "Experimental Values=<value>" (empty if NaN)
    other_info = selected["Experimental Values"].map(
        lambda x: f"Experimental Values={x}" if pd.notna(x) else ""
    )

    # Assemble output frame
    out = pd.DataFrame({
        # temporary; reassigned after shuffle
        "question_ID": np.arange(1, len(selected) + 1, dtype=int),
        "compound.name_1": selected["compound.name_1"].values,
        "SMILES_1": selected["SMILES_1"].values,
        "compound.name_2": selected["compound.name_2"].values,
        "SMILES_2": selected["SMILES_2"].values,
        "OPTIONS": options_str,  # same for all rows
        "question_category": "mixture_similarity",
        "prompt.1": selected.apply(make_prompt1, axis=1),
        "prompt.2": selected.apply(make_prompt2, axis=1),
        "answer": selected["answer"].values,
        "other_info": other_info.values,  # renamed & formatted
    })

    # GLOBAL SHUFFLE of entire rows (not per-column)
    perm = rng.permutation(len(out))
    out = out.iloc[perm].reset_index(drop=True)

    # Reassign question_ID to match the new shuffled order
    out["question_ID"] = np.arange(1, len(out) + 1, dtype=int)

    # Ensure exact column order
    cols_order = [
        "question_ID", "compound.name_1", "SMILES_1", "compound.name_2", "SMILES_2",
        "OPTIONS", "question_category", "prompt.1", "prompt.2", "answer", "other_info"
    ]
    out = out[cols_order]

    # Save
    out.to_csv(OUT_CSV, index=False)
    print(f"Wrote {OUT_CSV} with {len(out)} rows "
          f"(25 per class; CID_count threshold n={n_extra}). Rows globally shuffled.")

if __name__ == "__main__":
    main()


Achieved 25 per class with n = 0 (max CID_count allowed = 10).
Wrote mixture_similarity_100.csv with 100 rows (25 per class; CID_count threshold n=0). Rows globally shuffled.
